# Identifier une molécule à partir de ses spectres de masse (CASMI 2026)

## 1. Problème et démarche

Pour chaque molécule inconnue, on dispose d'un ou plusieurs spectres MS/MS et on propose jusqu'à 25 SMILES classés. Le score est le MRR@25 : la moyenne de 1/rang de la bonne structure, 0 si elle n'est pas dans les 25. Une structure est juste si les 14 premiers caractères de son InChIKey sont les bons ; la stéréochimie ne compte pas.

Un classifieur à une classe par molécule ne reconnaîtrait que des molécules déjà vues. On prédit donc un fingerprint de Morgan, puis on cherche les structures proches dans un catalogue :

1. masse neutre, à partir de `precursor_mz` et de l'`adduct` ;
2. candidats du catalogue à ±10 ppm ;
3. fingerprint prédit à partir du spectre (seule étape apprise) ;
4. tri des candidats par similarité de Tanimoto, et top 25.

La validation porte sur des molécules absentes de l'entraînement. Le test visible n'y sert pas, car ses spectres sont des copies de spectres du train.

### Imports et chemins

Les données sont lues dans `data/raw/` en local et dans `/kaggle/input/` sur Kaggle. Les fichiers produits vont dans `data/rapport/`, ce qui évite d'écraser ceux de l'ancien code. La cellule doit afficher les trois fichiers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SEED = 42

KAGGLE_INPUT = Path("/kaggle/input")
if KAGGLE_INPUT.exists():
    RAW_DIR = next(KAGGLE_INPUT.rglob("train.parquet")).parent
    WORK_DIR = Path("/kaggle/working")
else:
    RAW_DIR = Path("data/raw")
    WORK_DIR = Path("data/rapport")
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DIR / "train.parquet"
TEST_PATH = RAW_DIR / "test.parquet"
SUBMISSION_PATH = RAW_DIR / "sample_submission.csv"

for path in [TRAIN_PATH, TEST_PATH, SUBMISSION_PATH]:
    print(f"{path}  {path.stat().st_size / 1e6:,.1f} Mo")

## 2. Données

On charge seulement les colonnes utiles du train, sans les pics, qui sont les plus lourds. L'index de `train` est la position du spectre dans le fichier : il servira à relire les pics d'un échantillon. À regarder : le nombre de spectres et de molécules.

Constat : le train compte 2 539 608 spectres pour 275 810 molécules et tient en 0,7 Go sans les pics. Le test visible compte 1 213 spectres pour 400 molécules.

In [ ]:
COLUMNS = ["inchikey14", "normalized_smiles", "ingest_lib", "adduct", "ionization_mode",
           "instrument_type", "precursor_mz", "collision_energy_ev", "num_peaks"]

train = pd.read_parquet(TRAIN_PATH, columns=COLUMNS)
test = pd.read_parquet(TEST_PATH)

print(f"train : {len(train):,} spectres, {train['inchikey14'].nunique():,} molécules, "
      f"{train.memory_usage(deep=True).sum() / 1e9:.1f} Go en mémoire")
print(f"test  : {len(test):,} spectres, {test['molecule_id'].nunique():,} molécules")
train.head(3)

### Provenance des spectres

Le train réunit plusieurs bibliothèques de spectres (`ingest_lib`), mesurées dans des laboratoires et sur des appareils différents. Le graphique compte les spectres de chacune, et le tableau ajoute leur part et leur nombre de molécules. À regarder : le poids de la première bibliothèque par rapport aux autres.

Constat : enveda-180 fournit 45,4 % des spectres et 182 941 molécules. Les quatre plus grosses bibliothèques réunissent 88,6 % des spectres.

In [ ]:
counts = train["ingest_lib"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(counts.index, counts.values / 1e3, color="#2a78d6", height=0.7)
ax.set_xlabel("spectres (milliers)")
ax.set_title("Spectres du train par bibliothèque", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

molecules = train.groupby("ingest_lib")["inchikey14"].nunique()
pd.DataFrame({"spectres": counts, "% des spectres": (counts / len(train) * 100).round(1),
              "molécules": molecules}).sort_values("spectres", ascending=False)

### Valeurs manquantes

Dans `instrument_type`, certains spectres portent un texte (`"nan"`, `"N/A"` ou une chaîne vide) à la place d'une valeur manquante, et `isna()` ne les voit pas. On les convertit d'abord. Le tableau donne ensuite la part de valeurs manquantes des colonnes que nous utilisons, dans le train et dans le test.

Constat : 1 645 faux manquants convertis. Dans le train, `instrument_type` manque pour 1,37 % des spectres et `collision_energy_ev` pour 13,29 %. Le test visible n'a aucune valeur manquante.

In [ ]:
FAKE_NA = ["nan", "N/A", ""]  # textes mis à la place d'une valeur manquante
n_fake = train["instrument_type"].isin(FAKE_NA).sum()
train["instrument_type"] = train["instrument_type"].mask(train["instrument_type"].isin(FAKE_NA))
print(f"Faux manquants convertis dans instrument_type : {n_fake:,}")

cols = ["adduct", "ionization_mode", "precursor_mz", "instrument_type", "collision_energy_ev"]
display(pd.DataFrame({"train (%)": train[cols].isna().mean() * 100,
                      "test (%)": test[cols].isna().mean() * 100}).round(2))

La carte ci-dessous couvre les 18 colonnes du train. Les colonnes hors pics sont lues une par une ; pour les deux colonnes de pics, trop lourdes, on lit le nombre de valeurs nulles dans les métadonnées du fichier. Chaque ligne est une bibliothèque, puis viennent le train entier et le test visible ; « absente » signale une colonne que le test n'a pas. À regarder : si les manques sont répartis partout ou concentrés dans quelques bibliothèques, et ce qu'il en est dans le test.

Constat : cinq colonnes ont des valeurs manquantes, et tout dépend de la bibliothèque. enveda-180 et enveda-np-examples sont complètes, alors que sept bibliothèques ne renseignent jamais `base_peak_intensity`. Les pics n'ont aucune valeur nulle.

In [ ]:
pf = pq.ParquetFile(TRAIN_PATH)
peak_cols = ["ms2_mzs", "ms2_normalized_intensities"]
other_cols = [c for c in pf.schema_arrow.names if c not in peak_cols]

# Colonnes hors pics : lues une par une, pour ne jamais tout charger en mémoire
missing = pd.DataFrame({c: (train[c] if c in train else pd.read_parquet(TRAIN_PATH, columns=[c])[c]).isna()
                        for c in other_cols})
# Pics : trop lourds à lire, on compte les valeurs nulles dans les métadonnées du fichier
meta = pf.metadata
for c in peak_cols:
    i = [meta.schema.column(k).path.split(".")[0] for k in range(meta.num_columns)].index(c)
    print(f"{c} : {sum(meta.row_group(g).column(i).statistics.null_count for g in range(meta.num_row_groups))} valeur nulle")

order = train["ingest_lib"].value_counts().index
table = (missing.groupby(train["ingest_lib"]).mean() * 100).loc[order]
table.loc["train entier"] = missing.mean() * 100
table.loc["test visible"] = test.isna().mean() * 100  # NaN si la colonne n'existe pas dans le test

fig, ax = plt.subplots(figsize=(11, 5))
ax.imshow(table.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=100, aspect="auto")
for (i, j), v in np.ndenumerate(table.to_numpy(dtype=float)):
    label = "absente" if np.isnan(v) else (f"{v:.0f}" if v >= 0.5 else ("<1" if v > 0 else ""))
    ax.text(j, i, label, ha="center", va="center", fontsize=8, color="white" if v > 50 else "#52514e")
ax.set_xticks(range(table.shape[1]), table.columns, rotation=40, ha="right")
ax.set_yticks(range(table.shape[0]), table.index)
ax.axhline(len(order) - 0.5, color="#52514e", linewidth=1)
ax.set_title("Valeurs manquantes (% des spectres) par bibliothèque et par colonne", loc="left")
plt.tight_layout()
plt.show()

### Peut-on imputer les valeurs manquantes ?

On teste deux colonnes : `base_peak_intensity` (en log10, car elle varie sur plusieurs ordres de grandeur) et `collision_energy_ev` (la moyenne de la liste d'énergies). Les essais portent sur 500 000 spectres tirés au hasard, pour rester rapides. Les entrées des imputeurs n'utilisent que des colonnes présentes dans le test ; `instrument_type` est regroupé en familles d'appareils, car ses libellés sont très hétérogènes. À regarder : la répartition des familles.

Constat : timsTOF (226 717 spectres), Orbitrap/FT (187 309) et QTOF/TOF (67 209) dominent l'échantillon. 6 870 spectres n'ont pas d'instrument (« Inconnu »).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Familles d'instruments : la première dont le motif apparaît dans le libellé l'emporte
INSTRUMENT_FAMILIES = [
    ("timsTOF", r"tims"),
    ("Orbitrap/FT", r"orbitrap|exactive|qehf|ft\b|qft|itft|hybrid ft"),
    ("QTOF/TOF", r"tof|maxis|impact|synapt"),
    ("Trappe ionique", r"ion trap|iontrap|\blit\b|qit|ltq|-it\b"),
    ("Quadripôle", r"qqq|qq\b|quattro|-q\b"),
]

def instrument_family(s):
    """Famille de chaque libellé d'instrument ; "Inconnu" s'il manque, "Autre" s'il n'est d'aucune famille."""
    lower = s.str.lower()
    family = pd.Series("Autre", index=s.index).mask(s.isna(), "Inconnu")
    for name, pattern in reversed(INSTRUMENT_FAMILIES):
        family = family.mask(lower.str.contains(pattern, regex=True, na=False), name)
    return family

sample = train.sample(500_000, random_state=SEED)
top_adducts = train["adduct"].value_counts().index[:15]
X = pd.DataFrame({
    "precursor_mz": sample["precursor_mz"],
    "log_num_peaks": np.log10(sample["num_peaks"]),
    "mode": sample["ionization_mode"].astype("category"),
    "adduct": sample["adduct"].where(sample["adduct"].isin(top_adducts), "autre").astype("category"),
    "instrument": instrument_family(sample["instrument_type"]).astype("category"),
})
bpi = pd.read_parquet(TRAIN_PATH, columns=["base_peak_intensity"])["base_peak_intensity"]
y_bpi = np.log10(bpi.loc[sample.index])
y_ce = sample["collision_energy_ev"].dropna().map(np.mean).reindex(sample.index)  # moyenne de la liste d'énergies
X["instrument"].value_counts()

On masque des valeurs **observées**, on les prédit, et on mesure l'erreur. Deux masquages :
- au hasard, 20 % des valeurs ;
- une bibliothèque entière à la fois, ce qui reproduit les vrais manques, où une bibliothèque ne renseigne jamais la colonne.

Trois imputeurs, du plus simple au plus riche : la médiane globale, la médiane par famille d'instrument et mode, et un gradient boosting. La cellule prend une dizaine de secondes. À regarder : si l'erreur reste acceptable quand on cache une bibliothèque entière.

Constat :
- pour `base_peak_intensity`, le gradient boosting se trompe d'un facteur 10^0,68 ≈ 5 quand on masque au hasard. Quand on cache gnps, l'erreur monte à 1,95 en log10, soit un facteur 90 environ, et aucun imputeur ne fait mieux que la médiane globale (1,93) ;
- pour `collision_energy_ev`, la médiane globale est la plus mauvaise sur tous les masquages. La médiane par instrument et mode est la meilleure sur enveda-180 (8,99 eV) et pluskal_ms2 (12,70 eV), et le gradient boosting gagne sur les autres, avec des écarts de 1 à 4 eV.

In [ ]:
def imputation_trial(X, y, libs):
    """Erreur absolue moyenne de trois imputeurs sur des valeurs observées que l'on masque."""
    observed = y.notna()
    masks = {"au hasard (20 %)": observed & (np.random.default_rng(SEED).random(len(y)) < 0.2)}
    for lib, n in libs[observed].value_counts().items():
        if n >= 1000:
            masks[f"bibliothèque {lib}"] = observed & (libs == lib)
    rows = []
    for name, masked in masks.items():
        fit = observed & ~masked  # les imputeurs n'apprennent que sur les valeurs restées visibles
        median = y[fit].median()
        by_group = y[fit].groupby([X.loc[fit, "instrument"], X.loc[fit, "mode"]], observed=True).median()
        group_pred = X[masked].join(by_group.rename("pred"), on=["instrument", "mode"])["pred"].fillna(median)
        model = HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=SEED)
        model_pred = model.fit(X[fit], y[fit]).predict(X[masked])
        rows.append({"masquage": name, "valeurs masquées": masked.sum(),
                     "médiane globale": (y[masked] - median).abs().mean(),
                     "médiane par instrument et mode": (y[masked] - group_pred).abs().mean(),
                     "gradient boosting": (y[masked] - model_pred).abs().mean()})
    return pd.DataFrame(rows).set_index("masquage").round(2)

libs = sample["ingest_lib"]
print("base_peak_intensity : erreur absolue moyenne en log10 (1 = un facteur 10)")
display(imputation_trial(X, y_bpi, libs))
print("collision_energy_ev : erreur absolue moyenne en eV")
display(imputation_trial(X, y_ce, libs))

### Décision pour chaque colonne incomplète

On garde tous les spectres. Retirer ceux qui ont une valeur manquante supprimerait des bibliothèques entières, comme gnps, qui n'a aucune énergie de collision.

| Colonne | Manquants (train) | Décision | Raison |
|---|---|---|---|
| `base_peak_intensity` | 46 % | colonne écartée | Sept bibliothèques ne la renseignent jamais, donc son absence révèle surtout la bibliothèque. Les essais ci-dessus échouent quand on masque une bibliothèque entière. |
| `collision_energy_ev` | 13 % | médiane par famille d'instrument et mode, avec une variable indicatrice `ce_manquante` | Elle fait mieux que la médiane globale sur tous les masquages et reste proche du gradient boosting, pour un code bien plus simple. L'indicatrice dit au modèle que la valeur est estimée. |
| `collision_energy_orig` | 13 % | colonne écartée | C'est le texte d'origine de l'énergie, déjà convertie en eV dans `collision_energy_ev`. |
| `instrument_type` | 1 % | catégorie « Inconnu » | Les manques sont rares, et une catégorie à part évite de deviner l'appareil. |
| `precursor_error_ppm` | moins de 1 % | colonne écartée | Elle est calculée à partir de la vraie structure et n'existe pas dans le test : l'utiliser serait une fuite. |

Les médianes seront calculées sur les spectres d'entraînement seulement, après le split, puis appliquée à la validation et au test. Ces choix servent surtout à l'entraînement, puisque le test visible n'a aucune valeur manquante.